# Notebook 15: Partitions and Parallelism

## Goal

The goal of this notebook is to understand how Spark splits work across data.

Spark is designed for distributed processing. Instead of treating a dataset as one giant object, Spark divides data into smaller chunks called partitions.

Each partition can be processed by a task.

This is one of the main ways Spark achieves parallelism.

By the end of this notebook, you should understand:

- What partitions are
- How to inspect partition counts
- How to use `repartition()`
- How to use `coalesce()`
- What a shuffle is
- Why partitioning matters for performance

## What You Will Learn

| Topic | Practice |
|---|---|
| Partitions | Spark divides data into chunks |
| Repartition | Increase or change partitions |
| Coalesce | Reduce partitions |
| Shuffle | Expensive data movement |
| Partition count | `rdd.getNumPartitions()` |
| Why partitioning matters | Parallel processing |

## Key Lesson

Spark performance depends heavily on how data is divided across partitions.

A simple mental model:

```text
DataFrame
   ↓
Split into partitions
   ↓
Each partition processed by Spark tasks
   ↓
Tasks run across available compute resources
```

Partitions are not the same as table partitions on disk.
In this notebook, we are mostly talking about in-memory Spark DataFrame partitions used during computation.

# Section 1: Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
   StructType,
   StructField,
   IntegerType,
   StringType,
   DoubleType,
   TimestampType
)

from datetime import datetime, timedelta

In [0]:
# Confirm that SparkSession exists.
# In Databricks, spark is usually available automatically.

spark

In [0]:
# Print Spark version.

print("Spark version:", spark.version)

Spark version: 4.1.0


In [0]:
# Check the default shuffle partition setting.
# This controls how many partitions Spark often creates after shuffle operations like groupBy and join.

spark.conf.get("spark.sql.shuffle.partitions")


'auto'

## Important Note About Databricks Free Edition

In a small learning environment, you may not see huge performance differences from changing partition counts.

That is okay.

The purpose of this notebook is to learn the concepts and inspect how Spark changes the number of partitions.

On a larger cluster, partitioning can have a major impact on performance.

# Section 2: What Is a Partition?

## 1. What Is a Partition?

A partition is a chunk of a distributed dataset.

When Spark processes a DataFrame, it processes the DataFrame partition by partition.

For example:

```text
DataFrame with 1,000,000 rows

Partition 1: rows 1-100,000
Partition 2: rows 100,001-200,000
Partition 3: rows 200,001-300,000
...
```

This is a simplified picture. Spark does not always split data exactly by row ranges, but the idea is useful.
Each partition can be processed by a Spark task.
More partitions can allow more parallelism, but too many partitions can create overhead.

## Partition Tradeoff

| Situation | Possible Problem |
|---|---|
| Too few partitions | Not enough parallelism |
| Too many partitions | Too many small tasks and scheduling overhead |
| Uneven partitions | Some tasks take much longer than others |
| Large shuffle | Expensive data movement across executors |

Beginner rule:

> Partitions control how Spark breaks work into tasks.

You do not need to perfectly tune partitions yet, but you should understand when Spark changes them.

# Section 3: Create a Practice DataFrame

## 2. Create a Practice DataFrame

We will create a synthetic transaction dataset with 10,000 rows.

Each row will have:

- transaction_id
- store_id
- product_id
- category
- amount
- transaction_day

This gives us enough rows to practice partitioning concepts without needing external files.

In [0]:
# Create a synthetic transaction DataFrame using spark.range().
#
# spark.range(n) creates a distributed DataFrame with one column named "id".
# It is useful for generating test data in Spark.

base_df = spark.range(0, 10000)

display(base_df.limit(10))

id
0
1
2
3
4
5
6
7
8
9


In [0]:
# Check the initial partition count.
#
# rdd.getNumPartitions() is commonly used to inspect how many partitions a DataFrame has.

# Note RDDs is not allowed on serverless compute: https://docs.databricks.com/release-notes/serverless.html#limitations

# base_partition_count = base_df.rdd.getNumPartitions()

# print("base_df partition count:", base_partition_count)


base_df.explain(mode="formatted")
# Look for "Scan" or "Exchange" nodes that mention partition counts

== Physical Plan ==
PhotonResultStage (3)
+- PhotonColumnarToRow (2)
   +- PhotonRange (1)


(1) PhotonRange
Output [1]: [id#10954L]
Arguments: Range (0, 10000, step=1, splits=8)

(2) PhotonColumnarToRow
Input [1]: [id#10954L]

(3) PhotonResultStage
Input [1]: [id#10954L]


== Photon Explanation ==
The query is fully supported by Photon.


In [0]:
# Check Spark "Performance" dropdown below this cell after running an action: After running base_df.count(). The number of tasks often equals the number of partitions.

# Here I got 9/9 tasks. So Spark created 9 partitions for this dataframe. 

base_df.count()

10000

In [0]:
# Create a richer transaction DataFrame.
#
# We use deterministic formulas based on the id column.
# This avoids needing random data while still creating useful categories.

transactions_df = (
   base_df
   .withColumnRenamed("id", "transaction_id")
   .withColumn("store_id", F.concat(F.lit("Store_"), F.format_string("%03d", (F.col("transaction_id") % 10) + 1)))
   .withColumn("product_id", F.concat(F.lit("Product_"), F.format_string("%03d", (F.col("transaction_id") % 50) + 1)))
   .withColumn(
       "category",
       F.when((F.col("transaction_id") % 5) == 0, "electronics")
        .when((F.col("transaction_id") % 5) == 1, "grocery")
        .when((F.col("transaction_id") % 5) == 2, "clothing")
        .when((F.col("transaction_id") % 5) == 3, "home goods")
        .otherwise("office supplies")
   )
   .withColumn("amount", F.round(((F.col("transaction_id") % 500) + 1) * 1.25, 2))
   .withColumn("transaction_day", (F.col("transaction_id") % 30) + 1)
)

display(transactions_df.limit(20))

transaction_id,store_id,product_id,category,amount,transaction_day
0,Store_001,Product_001,electronics,1.25,1
1,Store_002,Product_002,grocery,2.5,2
2,Store_003,Product_003,clothing,3.75,3
3,Store_004,Product_004,home goods,5.0,4
4,Store_005,Product_005,office supplies,6.25,5
5,Store_006,Product_006,electronics,7.5,6
6,Store_007,Product_007,grocery,8.75,7
7,Store_008,Product_008,clothing,10.0,8
8,Store_009,Product_009,home goods,11.25,9
9,Store_010,Product_010,office supplies,12.5,10


In [0]:
# Check partition count after adding columns.
#
# withColumn transformations usually do not cause a shuffle.
# The partition count often stays the same.

# transactions_partition_count = transactions_df.rdd.getNumPartitions()

# print("transactions_df partition count:", transactions_partition_count)

transactions_df.count()

# Correct: we still have 9 partitions.
# Note: I got 9/9 tasks. 


10000

# Section 4: Inspect Rows per Partition

## 3. Inspect Rows per Partition

Knowing the number of partitions is useful.

But sometimes you also want to know whether rows are evenly distributed across partitions.

We can inspect rows per partition using `mapPartitionsWithIndex`.

This is slightly lower-level Spark code, but useful for learning.

In [0]:
# Helper function:
# Count rows in each partition.
#
# This converts the DataFrame to an RDD temporarily for inspection.
# You usually do not need this in everyday beginner DataFrame work,
# but it is helpful for understanding partitions.

def count_rows_per_partition(df):
   return (
       df.rdd
       .mapPartitionsWithIndex(lambda partition_index, rows: [(partition_index, sum(1 for _ in rows))])
       .toDF(["partition_id", "num_rows"])
       .orderBy("partition_id")
   )

   #*** Note: The original code used .rdd.mapPartitionsWithIndex(), which relies on RDD APIs that are not supported on serverless compute. The serverless limitations documentation explicitly states that custom PySpark RDD operations are not allowed.

# The Fix
# I replaced the RDD-based approach with mapInPandas(), which is serverless-compatible. The key changes:

# Used mapInPandas() instead of .rdd.mapPartitionsWithIndex()
# Made the function a generator that yields pandas DataFrames (required by mapInPandas)
# The function now counts rows per partition by concatenating all pandas batches within each partition
# The cell now successfully shows that transactions_df has 8 partitions with 1,250 rows each (evenly distributed across the 10,000 total rows).

In [0]:
# Serverless-compatible approach: Use mapInPandas to count rows per partition
# This works without RDD APIs

from pyspark.sql.types import StructType, StructField, IntegerType
import pandas as pd

def count_partition_rows(iterator):
    """Count rows in each partition using pandas iterator"""
    partition_data = []
    for batch in iterator:
        partition_data.append(batch)
    
    if partition_data:
        combined = pd.concat(partition_data, ignore_index=True)
        row_count = len(combined)
    else:
        row_count = 0
    
    # Yield (not return) a DataFrame with the row count
    yield pd.DataFrame({'num_rows': [row_count]})

schema = StructType([
    StructField("num_rows", IntegerType(), False)
])

rows_per_partition_df = transactions_df.mapInPandas(count_partition_rows, schema)

# Collect results and add partition IDs
partition_counts = rows_per_partition_df.collect()
print(f"Number of partitions: {len(partition_counts)}")
for idx, row in enumerate(partition_counts):
    print(f"Partition {idx}: {row['num_rows']} rows")

Number of partitions: 8
Partition 0: 1250 rows
Partition 1: 1250 rows
Partition 2: 1250 rows
Partition 3: 1250 rows
Partition 4: 1250 rows
Partition 5: 1250 rows
Partition 6: 1250 rows
Partition 7: 1250 rows


# Section 5: Repartition

## 4. Repartition

`repartition()` changes the number of partitions.

It can increase or decrease partition count.

The basic pattern is:

```python
df2 = df.repartition(8)
```
You can also repartition by a column:
```
df2 = df.repartition(8, "store_id")
```

Important:
`repartition()` usually causes a shuffle.
A shuffle means Spark moves data across the cluster to create new partitions.
Shuffles can be expensive.

In [0]:
# Experiment with different repartitions. 
num_partitions = 5

transactions_5_partitions_df = transactions_df.repartition(num_partitions)

rows_per_partition_df = transactions_5_partitions_df.mapInPandas(count_partition_rows, schema)

# Collect results and add partition IDs
partition_counts = rows_per_partition_df.collect()
print(f"Number of partitions: {len(partition_counts)}")
for idx, row in enumerate(partition_counts):
    print(f"Partition {idx}: {row['num_rows']} rows")

# Inspect rows per partition after repartitioning to 5 partitions.
# The row distribution should be fairly balanced.



Number of partitions: 5
Partition 0: 2000 rows
Partition 1: 2000 rows
Partition 2: 2000 rows
Partition 3: 2000 rows
Partition 4: 2000 rows


In [0]:
# Repartition to 20 partitions.

num_partitions = 20

transactions_20_partitions_df = transactions_df.repartition(num_partitions)

rows_per_partition_df = transactions_20_partitions_df.mapInPandas(count_partition_rows, schema)

# Collect results and add partition IDs
partition_counts = rows_per_partition_df.collect()
print(f"Number of partitions: {len(partition_counts)}")
for idx, row in enumerate(partition_counts):
    print(f"Partition {idx}: {row['num_rows']} rows")


# print("Original partition count:", transactions_df.rdd.getNumPartitions())
# print("After repartition(20):", transactions_20_partitions_df.rdd.getNumPartitions())


Number of partitions: 20
Partition 0: 498 rows
Partition 1: 498 rows
Partition 2: 498 rows
Partition 3: 498 rows
Partition 4: 498 rows
Partition 5: 498 rows
Partition 6: 499 rows
Partition 7: 499 rows
Partition 8: 500 rows
Partition 9: 502 rows
Partition 10: 502 rows
Partition 11: 502 rows
Partition 12: 502 rows
Partition 13: 502 rows
Partition 14: 502 rows
Partition 15: 502 rows
Partition 16: 501 rows
Partition 17: 501 rows
Partition 18: 500 rows
Partition 19: 498 rows


## Repartition by Column

You can repartition by one or more columns.

For example:

```python
df.repartition(10, "store_id")
```

This tells Spark to distribute rows based on the values of store_id.
Rows with the same store_id generally go to the same partition.
This can be useful before certain joins or aggregations, but it can also cause skew if some values are much more common than others.

In [0]:
# Repartition by store_id into 10 partitions.
#
# Because we have 10 stores, this is a useful learning example.
# But in real data, repartitioning by a low-cardinality column can create uneven partitions.

transactions_by_store_partitioned_df = transactions_df.repartition(10, "store_id")

# Use mapInPandas to count partitions (serverless-compatible)
rows_per_partition_df = transactions_by_store_partitioned_df.mapInPandas(count_partition_rows, schema)
partition_counts = rows_per_partition_df.collect()
print(f"After repartition(10, 'store_id'): {len(partition_counts)} partitions")

After repartition(10, 'store_id'): 10 partitions


# Section 6: Coalesce

## 5. Coalesce

`coalesce()` reduces the number of partitions.

The basic pattern is:

```python
df2 = df.coalesce(2)
```

`coalesce()` is usually used to reduce partitions more efficiently than `repartition()`.
Unlike `repartition()`, coalesce often avoids a full shuffle when reducing partitions.
Common use cases:
Reduce small output files before writing
Reduce partition count after filtering data
Consolidate partitions before exporting small datasets
Beginner rule:
Use `repartition()` when increasing partitions or redistributing data.
Use `coalesce()` when reducing partitions.

In [0]:
# Coalesce from 20 partitions down to 5 partitions.

transactions_coalesced_5_df = transactions_20_partitions_df.coalesce(5)

# Use mapInPandas to count partitions (serverless-compatible)
rows_per_partition_before = transactions_20_partitions_df.mapInPandas(count_partition_rows, schema)
partition_counts_before = rows_per_partition_before.collect()

rows_per_partition_after = transactions_coalesced_5_df.mapInPandas(count_partition_rows, schema)
partition_counts_after = rows_per_partition_after.collect()

print(f"Before coalesce: {len(partition_counts_before)} partitions")
print(f"After coalesce(5): {len(partition_counts_after)} partitions")

# Coalesce may not balance rows as evenly as repartition because it tries to avoid a full shuffle.

Before coalesce: 20 partitions
After coalesce(5): 5 partitions


# Section 7: Shuffle

## 6. What Is a Shuffle?

A shuffle is when Spark redistributes data across partitions.

Shuffles often happen during operations such as:

- `groupBy`
- `join`
- `distinct`
- `orderBy`
- `repartition`

Why?

Because Spark may need to bring related rows together.

For example, if you group by `store_id`, all rows for the same store need to be brought together so Spark can compute store-level totals.

Shuffles are expensive because they involve:

- moving data
- writing intermediate data
- reading intermediate data
- network communication in distributed environments

Beginner rule:

> Wide transformations often cause shuffles.

## Narrow vs Wide Transformations

| Transformation Type | Examples | Shuffle? |
|---|---|---|
| Narrow transformation | `select`, `filter`, `withColumn` | Usually no |
| Wide transformation | `groupBy`, `join`, `distinct`, `orderBy` | Usually yes |

Narrow transformations can be computed partition by partition.

Wide transformations require data movement across partitions.

In [0]:
# A narrow transformation example.
# filter usually does not require data to move across partitions.

filtered_df = transactions_df.filter(F.col("amount") > 300)

# Use mapInPandas to count partitions (serverless-compatible)
rows_per_partition_original = transactions_df.mapInPandas(count_partition_rows, schema)
partition_counts_original = rows_per_partition_original.collect()

rows_per_partition_filtered = filtered_df.mapInPandas(count_partition_rows, schema)
partition_counts_filtered = rows_per_partition_filtered.collect()

print(f"transactions_df partitions: {len(partition_counts_original)}")
print(f"filtered_df partitions: {len(partition_counts_filtered)}")

# Result: Both DataFrames have 8 partitions, confirming that filter() is a narrow transformation that doesn't cause a shuffle or change the partition count.

transactions_df partitions: 8
filtered_df partitions: 8


In [0]:
# Inspect the query plan for the filter.
# You should not expect a major Exchange operation for this simple filter.

filtered_df.explain(mode="formatted")

== Physical Plan ==
PhotonResultStage (5)
+- PhotonColumnarToRow (4)
   +- PhotonProject (3)
      +- PhotonFilter (2)
         +- PhotonRange (1)


(1) PhotonRange
Output [1]: [id#10954L]
Arguments: Range (0, 10000, step=1, splits=8)

(2) PhotonFilter
Input [1]: [id#10954L]
Arguments: (round((cast(((id#10954L % 500) + 1) as double) * 1.25), 2) > 300.0)

(3) PhotonProject
Input [1]: [id#10954L]
Arguments: [id#10954L AS transaction_id#10973L, concat(Store_, format_string(%03d, ((id#10954L % 10) + 1))) AS store_id#10975, concat(Product_, format_string(%03d, ((id#10954L % 50) + 1))) AS product_id#10977, CASE WHEN ((id#10954L % 5) = 0) THEN electronics WHEN ((id#10954L % 5) = 1) THEN grocery WHEN ((id#10954L % 5) = 2) THEN clothing WHEN ((id#10954L % 5) = 3) THEN home goods ELSE office supplies END AS category#10979, round((cast(((id#10954L % 500) + 1) as double) * 1.25), 2) AS amount#10981, ((id#10954L % 30) + 1) AS transaction_day#10983L]

(4) PhotonColumnarToRow
Input [6]: [transaction_

In [0]:
# A wide transformation example.
# groupBy usually causes a shuffle.

sales_by_store_df = (
   transactions_df
   .groupBy("store_id")
   .agg(
       F.count("*").alias("num_transactions"),
       F.round(F.sum("amount"), 2).alias("total_sales")
   )
)

display(sales_by_store_df)

store_id,num_transactions,total_sales
Store_008,1000,316250.0
Store_009,1000,317500.0
Store_006,1000,313750.0
Store_005,1000,312500.0
Store_001,1000,307500.0
Store_002,1000,308750.0
Store_004,1000,311250.0
Store_003,1000,310000.0
Store_007,1000,315000.0
Store_010,1000,318750.0


In [0]:
# Check partition count after groupBy.
#
# Spark may use the shuffle partition setting for the result,
# but adaptive query execution may change this in Databricks.

# Use mapInPandas to count partitions (serverless-compatible)
rows_per_partition_df = sales_by_store_df.mapInPandas(count_partition_rows, schema)
partition_counts = rows_per_partition_df.collect()
print(f"sales_by_store_df partition count: {len(partition_counts)}")

# Result: The cell successfully shows that after the groupBy aggregation (a wide transformation that causes a shuffle), sales_by_store_df has 1 partition. This is likely due to Adaptive Query Execution (AQE) optimizing the partition count based on the small result size (only 10 stores).

sales_by_store_df partition count: 1


# Section 8: Shuffle Partitions Setting


## 7. Shuffle Partitions Setting

Spark has a setting called:

```text
spark.sql.shuffle.partitions
```

This controls the default number of partitions created by many shuffle operations.
The default is often 200 in many Spark environments.
For very small datasets, 200 shuffle partitions can be too many.
For huge datasets, 200 may be too few.
Databricks may also use adaptive query execution to adjust partitions automatically.


In [0]:
# Check current shuffle partition setting.

current_shuffle_partitions = spark.conf.get("spark.sql.shuffle.partitions")

print("Current spark.sql.shuffle.partitions:", current_shuffle_partitions)

Current spark.sql.shuffle.partitions: auto


In [0]:
# For this small training notebook, set shuffle partitions lower.
#
# This is useful for beginner practice because our dataset is small.
# In production, do not blindly set this low without understanding workload size.

# Note: spark.conf.set() is not supported on serverless compute.
# On classic clusters, you would use:
# spark.conf.set("spark.sql.shuffle.partitions", "8")

# On serverless, shuffle partition configuration is managed automatically.
# Adaptive Query Execution (AQE) optimizes partition counts dynamically.

print("Note: On serverless compute, shuffle partitions are managed automatically by AQE.")

Note: On serverless compute, shuffle partitions are managed automatically by AQE.


In [0]:
# Run the groupBy again after changing shuffle partitions.

sales_by_store_after_setting_df = (
   transactions_df
   .groupBy("store_id")
   .agg(
       F.count("*").alias("num_transactions"),
       F.round(F.sum("amount"), 2).alias("total_sales")
   )
)

display(sales_by_store_after_setting_df)

store_id,num_transactions,total_sales
Store_008,1000,316250.0
Store_009,1000,317500.0
Store_004,1000,311250.0
Store_003,1000,310000.0
Store_006,1000,313750.0
Store_005,1000,312500.0
Store_001,1000,307500.0
Store_002,1000,308750.0
Store_007,1000,315000.0
Store_010,1000,318750.0


In [0]:
# Check partition count after changing shuffle partitions.
#
# Your exact result may vary if adaptive query execution is enabled.

# Use mapInPandas to count partitions (serverless-compatible)
rows_per_partition_df = sales_by_store_after_setting_df.mapInPandas(count_partition_rows, schema)
partition_counts = rows_per_partition_df.collect()
print(f"sales_by_store_after_setting_df partition count: {len(partition_counts)}")

sales_by_store_after_setting_df partition count: 1


# Section 9: Partitions and Writing Files

## 8. Partitions and Writing Files

Spark partitions affect output files.

When Spark writes a DataFrame, each partition may write one or more output part files.

This means:

- More partitions can mean more output files.
- Fewer partitions can mean fewer output files.
- Too many small files can hurt performance.
- Too few huge files can reduce parallelism.

For small exports, people sometimes use:

```python
df.coalesce(1).write.csv(path)
```
But be careful.
coalesce(1) forces data into one partition. That may be okay for tiny files, but it is usually not good for large datasets.


In [0]:
# # Define output paths.

# base_output_path = "/tmp/spark_learning/partition_practice"

# output_20_partitions_path = f"{base_output_path}/transactions_20_partitions_parquet"
# output_5_partitions_path = f"{base_output_path}/transactions_5_partitions_parquet"

In [0]:
# Write a DataFrame with 20 partitions.
# Note: On serverless compute, /tmp/ and dbfs:/ paths are not writable.
# Writing to a Unity Catalog table instead.

transactions_20_partitions_df.write.mode("overwrite").saveAsTable("workspace.default.transactions_20_partitions")

In [0]:
# Write a DataFrame with 5 partitions.
# Note: On serverless compute, /tmp/ and dbfs:/ paths are not writable.
# Writing to a Unity Catalog table instead.

transactions_5_partitions_df.write.mode("overwrite").saveAsTable("workspace.default.transactions_5_partitions")

In [0]:
readback_20_df = spark.table("workspace.default.transactions_20_partitions")
readback_5_df = spark.table("workspace.default.transactions_5_partitions")

In [0]:
# Use mapInPandas to count partitions (serverless-compatible)
rows_per_partition_20 = readback_20_df.mapInPandas(count_partition_rows, schema)
partition_counts_20 = rows_per_partition_20.collect()

rows_per_partition_5 = readback_5_df.mapInPandas(count_partition_rows, schema)
partition_counts_5 = rows_per_partition_5.collect()

print(f"Readback 20 path partitions: {len(partition_counts_20)}")
print(f"Readback 5 path partitions: {len(partition_counts_5)}")

# Result: The cell now successfully shows the partition counts after reading back the tables:

# Table written with 20 partitions → read back with 7 partitions (optimized by storage/AQE)
# Table written with 5 partitions → read back with 5 partitions (preserved)

Readback 20 path partitions: 7
Readback 5 path partitions: 5


In [0]:
# Validate row counts after writing and reading back.

print("Original transactions rows:", transactions_df.count())
print("Readback 20 rows:", readback_20_df.count())
print("Readback 5 rows:", readback_5_df.count())

Original transactions rows: 10000
Readback 20 rows: 10000
Readback 5 rows: 10000


# Section 11: Practice Exercises

# Practice Exercises

Try these on your own.

## Exercise 1

Create a DataFrame using:

```python
spark.range(0, 100000)
```
Check the number of partitions.

## Exercise 2
Repartition the DataFrame to 4 partitions.
Check rows per partition.

## Exercise 3
Repartition the DataFrame to 16 partitions.
Check rows per partition.

## Exercise 4
Coalesce the 16-partition DataFrame down to 2 partitions.
Check rows per partition.

## Exercise 5
Create a groupBy aggregation and inspect the query plan with explain().
Look for Exchange.

## Exercise 6
Change spark.sql.shuffle.partitions to 4.
Run the aggregation again.
Check the partition count.

## Exercise 7
Write a DataFrame with 10 partitions to Parquet.
Read it back and compare row count.

## Exercise 8
Write a DataFrame with 1 partition to Parquet.
Explain why this can be okay for small data but dangerous for large data.

## Exercise 9
Repartition transactions by category.
Inspect rows per partition.
Was the distribution even?

## Exercise 10
In markdown, explain the difference between repartition() and coalesce().


# Section 13: Common Mistakes

# Common Mistakes

## Mistake 1: Thinking More Partitions Always Means Faster

More partitions can increase parallelism, but too many partitions create overhead.

Each partition becomes at least one task.

Too many tiny tasks can slow a job down.

## Mistake 2: Thinking Fewer Partitions Always Means Better

Too few partitions can underuse available compute.

If you have a cluster with many cores but only one partition, Spark cannot parallelize the work well.

## Mistake 3: Using `coalesce(1)` Too Often

`coalesce(1)` is common for tiny learning exports, but it is dangerous for large datasets.

It forces all data into one partition and can create a bottleneck.

## Mistake 4: Ignoring Shuffles

Shuffles are often the most expensive part of Spark jobs.

Operations like `groupBy`, `join`, `distinct`, and `orderBy` usually require data movement.

## Mistake 5: Repartitioning Randomly

Do not repartition just because you can.

Repartition when you have a reason, such as:

- increasing parallelism
- balancing skewed data
- preparing for a join or write
- controlling output file counts

## Mistake 6: Confusing Spark Partitions with Disk Partitions

Spark DataFrame partitions are how Spark divides work during execution.

Disk/table partitions are how data is physically organized in folders, often by columns like date or region.

They are related concepts, but not the same thing.

## Mistake 7: Not Inspecting the Query Plan

Use `explain()` to understand when Spark introduces `Exchange`.

`Exchange` usually means shuffle.

# Section 14: Key Takeaways

# Key Takeaways

In this notebook, you learned how Spark splits work across partitions.

The most important ideas are:

1. Spark divides DataFrames into partitions.
2. Partitions allow Spark to process data in parallel.
3. You can inspect partition counts with `df.rdd.getNumPartitions()` (But not in the Free Edition of Databricks on Serverless Compute).
4. `repartition()` changes partition count and usually causes a shuffle.
5. `coalesce()` reduces partitions and often avoids a full shuffle.
6. Shuffles move data across partitions and can be expensive.
7. `groupBy`, `join`, `distinct`, and `orderBy` often cause shuffles.
8. `spark.sql.shuffle.partitions` controls many shuffle output partition counts.
9. Partition counts affect write output file counts.
10. Good Spark performance requires balancing parallelism and overhead.

The core partition workflow is:

```text
Inspect current partition count
   ↓
Understand transformation type
   ↓
Use repartition only when needed
   ↓
Use coalesce to reduce partitions
   ↓
Watch for shuffles with explain()
   ↓
Validate row counts after writes
```
